# Podstawy sztucznej inteligencji: gry, minimax i alfa-beta

Ten notatnik jest przeznaczony do pracy w **Google Colab**.

## Cele
Po wykonaniu notatnika student powinien:
- umieć opisać grę jako problem przeszukiwania (stan, gracz, akcje, wynik, test końca, wypłata),
- rozumieć działanie algorytmu **minimax**,
- rozumieć, jak działają **cięcia alfa-beta** i dlaczego nie zmieniają wyniku,
- wiedzieć, dlaczego **kolejność ruchów** wpływa na liczbę odwiedzonych węzłów,
- umieć użyć **ograniczonej głębokości** i napisać własną **funkcję oceny**,
- potrafić porównać algorytmy pod względem liczby odwiedzonych węzłów i jakości gry.

## Plan
| Kiedy | Co |
|---|---|
| **Na zajęciach** | sekcje 1–3 (minimax, alfa-beta) i **Zadanie 1** |
| **Do domu** | sekcja 4 (cztery w rzędzie): **Zadanie 2** i **Zadanie 3** |
| *Dodatkowe* | zadania 4–7 |

Notatnik nawiązuje do laboratorium 1: tam algorytm A* dostawał funkcję `neighbors_fn` i działał dla siatki oraz 8-puzzle. Tu będzie tak samo: **jeden minimax działa dla każdej gry**, która ma te same funkcje.

## 1. Gra jako problem przeszukiwania

W laboratorium 1 problem opisywaliśmy przez stan początkowy, akcje, test celu i koszt.
W grze dwuosobowej potrzebujemy trochę innych elementów:

| Element | Znaczenie | Kółko i krzyżyk |
|---|---|---|
| `start` | stan początkowy | pusta plansza |
| `gracz(s)` | kto teraz rusza: `"MAX"` albo `"MIN"` | X to MAX, O to MIN |
| `akcje(s)` | dozwolone ruchy | wolne pola |
| `wynik(s, a)` | stan po wykonaniu ruchu `a` | plansza z nowym znakiem |
| `koniec(s)` | czy gra się skończyła? | trzy w linii lub pełna plansza |
| `wyplata(s)` | wynik na końcu gry (dla MAX) | +1, 0 albo −1 |

Stanem jest **krotka 9 pól** (jak w 8-puzzle z laboratorium 1). Pola numerujemy tak:

```text
0 | 1 | 2
3 | 4 | 5
6 | 7 | 8
```

`"."` oznacza wolne pole.

In [ ]:
import math
import random
import time
import matplotlib.pyplot as plt

random.seed(0)

LINIE_KK = [(0, 1, 2), (3, 4, 5), (6, 7, 8),   # wiersze
            (0, 3, 6), (1, 4, 7), (2, 5, 8),   # kolumny
            (0, 4, 8), (2, 4, 6)]              # przekątne


class KolkoKrzyzyk:
    start = ('.',) * 9

    def gracz(self, s):
        # X zaczyna; jeśli znaków X i O jest tyle samo, rusza X
        return "MAX" if s.count('X') == s.count('O') else "MIN"

    def akcje(self, s):
        return [i for i, pole in enumerate(s) if pole == '.']

    def wynik(self, s, a):
        znak = 'X' if self.gracz(s) == "MAX" else 'O'
        return s[:a] + (znak,) + s[a + 1:]

    def zwyciezca(self, s):
        for a, b, c in LINIE_KK:
            if s[a] != '.' and s[a] == s[b] == s[c]:
                return s[a]
        return None

    def koniec(self, s):
        return self.zwyciezca(s) is not None or '.' not in s

    def wyplata(self, s):
        z = self.zwyciezca(s)
        return 1 if z == 'X' else -1 if z == 'O' else 0

    def tekst(self, s):
        wiersze = [' '.join(s[r * 3:r * 3 + 3]) for r in range(3)]
        return '\n'.join(wiersze)


kk = KolkoKrzyzyk()


def z_napisu(napis):
    """Wygodny zapis pozycji: 'X...O...X' -> krotka 9 pól."""
    napis = napis.replace(' ', '').replace('\n', '')
    assert len(napis) == 9
    return tuple(napis)


s = kk.start
print(kk.tekst(s))
print("Rusza:", kk.gracz(s))
print("Akcje:", kk.akcje(s))
s = kk.wynik(s, 4)
print()
print(kk.tekst(s))
print("Rusza:", kk.gracz(s))

### Wizualizacja planszy

Funkcja `rysuj_kk` rysuje planszę. Opcjonalnie może wypisać liczby na wolnych polach (przyda się do pokazania wartości ruchów).

In [ ]:
def rysuj_kk(s, liczby=None, tytul=""):
    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    for i in (1, 2):
        ax.plot([i, i], [0, 3], color="black", lw=3)
        ax.plot([0, 3], [i, i], color="black", lw=3)
    for k, pole in enumerate(s):
        r, c = divmod(k, 3)
        x, y = c + 0.5, 2.5 - r
        if pole == 'X':
            ax.text(x, y, 'X', ha='center', va='center', fontsize=36, color="#0070C0", weight='bold')
        elif pole == 'O':
            ax.text(x, y, 'O', ha='center', va='center', fontsize=36, color="#FF0000", weight='bold')
        elif liczby is not None and k in liczby:
            ax.text(x, y, str(liczby[k]), ha='center', va='center', fontsize=18, color="#555555")
    ax.set_xlim(0, 3); ax.set_ylim(0, 3); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(tytul)
    plt.show()


rysuj_kk(z_napisu("X...O...X"), tytul="Pozycja z wykładu")

### Rozgrywka

Gracz to funkcja `gracz(gra, stan)`, która zwraca wybraną akcję. Najprostszy gracz wybiera losowo.
Funkcja `rozegraj` jest wspólna dla wszystkich gier w tym notatniku.

In [ ]:
def gracz_losowy(gra, s):
    return random.choice(gra.akcje(s))


def rozegraj(gra, gracz_max, gracz_min, pokaz=True):
    s = gra.start
    while not gra.koniec(s):
        ruch = gracz_max(gra, s) if gra.gracz(s) == "MAX" else gracz_min(gra, s)
        s = gra.wynik(s, ruch)
        if pokaz:
            print(gra.tekst(s), "\n")
    if pokaz:
        print("Wypłata dla MAX:", gra.wyplata(s))
    return gra.wyplata(s)


rozegraj(kk, gracz_losowy, gracz_losowy)

### Pytania

1. Dlaczego funkcja `gracz(s)` nie potrzebuje osobnej zmiennej „czyj ruch”? Skąd wie, kto rusza?
2. Ile akcji jest w stanie początkowym, a ile po trzech ruchach?
3. Czym `wyplata` różni się od kosztu z laboratorium 1? Kiedy ją znamy?

# 2. Minimax

Minimax liczy wartość stanu **od dołu drzewa do góry**:

- koniec gry: wartość = wypłata,
- ruch MAX: wartość = **maksimum** wartości następników,
- ruch MIN: wartość = **minimum** wartości następników.

To jest przeszukiwanie **w głąb** (DFS): rekurencja schodzi do końca gry, a wartości wracają w górę.

Dodajemy globalny `licznik`, który liczy odwiedzone węzły (każde wywołanie funkcji).

In [ ]:
licznik = 0


def minimax(gra, s):
    global licznik
    licznik += 1
    if gra.koniec(s):
        return gra.wyplata(s)
    wartosci = [minimax(gra, gra.wynik(s, a)) for a in gra.akcje(s)]
    if gra.gracz(s) == "MAX":
        return max(wartosci)
    return min(wartosci)


licznik = 0
t0 = time.time()
v = minimax(kk, kk.start)
print("Wartość pustej planszy:", v)
print("Odwiedzone węzły:", licznik)          # oczekiwane: 549 946
print(f"Czas: {time.time() - t0:.1f} s")

Wartość **0** oznacza remis: przy najlepszej grze obu stron nikt nie wygrywa. Liczba węzłów jest taka sama jak na wykładzie.

### Wartości ruchów

Funkcja `wartosci_ruchow` liczy minimax dla każdego możliwego ruchu. Sprawdźmy pozycję-pułapkę z wykładu: X zajął dwa przeciwległe rogi, O jest w środku, teraz ruch O.

In [ ]:
def wartosci_ruchow(gra, s, ocen=minimax):
    return {a: ocen(gra, gra.wynik(s, a)) for a in gra.akcje(s)}


pulapka = z_napisu("X...O...X")
w = wartosci_ruchow(kk, pulapka)
print("Rusza:", kk.gracz(pulapka))
print("Wartości ruchów:", w)
rysuj_kk(pulapka, liczby=w, tytul="+1: wygrywa X, 0: remis")

Ruch w róg wygląda dobrze, ale **przegrywa**: X stawia trzeci róg i ma dwie groźby naraz. O musi zagrać na bok.

### Gracz minimax

MAX wybiera ruch o największej wartości, MIN o najmniejszej. Przy remisie wartości losujemy, żeby partie były różne.

In [ ]:
def najlepszy_ruch(gra, s, ocen=minimax):
    w = wartosci_ruchow(gra, s, ocen)
    cel = max(w.values()) if gra.gracz(s) == "MAX" else min(w.values())
    return random.choice([a for a, v in w.items() if v == cel])


def gracz_minimax(gra, s):
    return najlepszy_ruch(gra, s)


# Minimax (X) przeciw graczowi losowemu (O). Pierwszy ruch trwa kilka sekund.
wyniki = [rozegraj(kk, gracz_minimax, gracz_losowy, pokaz=False) for _ in range(5)]
print("Wypłaty w 5 partiach:", wyniki)

### Zagraj z komputerem

Ustaw `GRAJ = True` i uruchom komórkę. Wpisuj numer pola (0–8). Spróbuj wygrać z minimaksem.

In [ ]:
GRAJ = False


def gracz_czlowiek(gra, s):
    print(gra.tekst(s))
    while True:
        try:
            a = int(input(f"Twój ruch {gra.akcje(s)}: "))
            if a in gra.akcje(s):
                return a
        except ValueError:
            pass
        print("Niepoprawny ruch.")


if GRAJ:
    rozegraj(kk, gracz_czlowiek, gracz_minimax)

### Pytania

1. Dlaczego minimax to w praktyce DFS? Ile pamięci potrzebuje?
2. Dlaczego minimax nigdy nie przegra w kółko i krzyżyk?
3. Czy minimax wygrywa z graczem losowym za każdym razem? Dlaczego nie zawsze?

# 3. Cięcia alfa-beta

Pomysł z wykładu: *jeśli wiem, że ruch jest gorszy, nie muszę wiedzieć, o ile gorszy.*

- **α (alfa)**: najlepsza wartość, którą MAX ma już zapewnioną,
- **β (beta)**: najlepsza wartość, którą MIN ma już zapewnioną,
- gdy **α ≥ β**, dalsze ruchy w tym węźle nic nie zmienią: **cięcie**.

Parametr `porzadek(s, akcje)` pozwala zmienić kolejność sprawdzania ruchów. Domyślnie sprawdzamy pola po kolei: 0, 1, …, 8.

In [ ]:
def alfabeta(gra, s, alfa=-math.inf, beta=math.inf, porzadek=None):
    global licznik
    licznik += 1
    if gra.koniec(s):
        return gra.wyplata(s)
    akcje = gra.akcje(s)
    if porzadek is not None:
        akcje = porzadek(s, akcje)
    if gra.gracz(s) == "MAX":
        v = -math.inf
        for a in akcje:
            v = max(v, alfabeta(gra, gra.wynik(s, a), alfa, beta, porzadek))
            if v >= beta:
                return v          # cięcie
            alfa = max(alfa, v)
        return v
    else:
        v = math.inf
        for a in akcje:
            v = min(v, alfabeta(gra, gra.wynik(s, a), alfa, beta, porzadek))
            if v <= alfa:
                return v          # cięcie
            beta = min(beta, v)
        return v


licznik = 0
t0 = time.time()
v = alfabeta(kk, kk.start)
print("Wartość pustej planszy:", v)
print("Odwiedzone węzły:", licznik)          # oczekiwane: 18 297
print(f"Czas: {time.time() - t0:.2f} s")

Wynik jest ten sam (remis), ale węzłów jest **30 razy mniej**.

### Kolejność ruchów

Alfa-beta tnie najwięcej, gdy **najpierw sprawdza najlepsze ruchy**. W kółku i krzyżyku dobre ruchy to zwykle środek, potem rogi, na końcu boki.

Funkcja `porzadek_losowy` jest przykładem: sprawdza ruchy w losowej kolejności. Uruchom komórkę kilka razy.

In [ ]:
def porzadek_losowy(s, akcje):
    return random.sample(akcje, len(akcje))


licznik = 0
alfabeta(kk, kk.start, porzadek=porzadek_losowy)
print("Losowa kolejność - odwiedzone węzły:", licznik)   # za każdym razem trochę inaczej

## Zadanie 1 — dobra kolejność ruchów

Napisz funkcję `porzadek_srodek_rogi(s, akcje)`, która zwraca akcje w kolejności: **środek → rogi → boki**.
Wewnątrz każdej grupy zachowaj kolejność rosnącą (np. rogi: 0, 2, 6, 8).

Podpowiedź: `sorted(akcje, key=...)` ze słownikiem priorytetów.

Jeśli wszystko jest dobrze, alfa-beta odwiedzi **7 275** węzłów.

In [ ]:
def porzadek_srodek_rogi(s, akcje):
    priorytet = {4: 0, 0: 1, 2: 1, 6: 1, 8: 1, 1: 2, 3: 2, 5: 2, 7: 2}
    return sorted(akcje, key=lambda a: (priorytet[a], a))


licznik = 0
v = alfabeta(kk, kk.start, porzadek=porzadek_srodek_rogi)
print("Wartość:", v, " odwiedzone węzły:", licznik)
assert v == 0 and licznik == 7275


### Porównanie

In [ ]:
wyniki = []
for nazwa, f in [("Minimax", None), ("Alfa-beta, pola 0-8", "brak"),
                 ("Alfa-beta, losowa kolejność", porzadek_losowy),
                 ("Alfa-beta, środek-rogi-boki", porzadek_srodek_rogi)]:
    licznik = 0
    if f is None:
        v = minimax(kk, kk.start)
    else:
        v = alfabeta(kk, kk.start, porzadek=None if f == "brak" else f)
    wyniki.append((nazwa, v, licznik))

print(f"{'Algorytm':<30} {'Wartość':<9} {'Węzły'}")
print("-" * 50)
for nazwa, v, n in wyniki:
    print(f"{nazwa:<30} {v:<9} " + f"{n:,}".replace(",", " "))

### Pytania

1. Dlaczego alfa-beta daje **zawsze ten sam wynik** co minimax?
2. Dlaczego kolejność ruchów zmienia liczbę węzłów, ale nie zmienia wyniku? (Porównaj z zadaniem 7 z laboratorium 1 o remisach w A*.)
3. Skąd w prawdziwej grze wziąć „dobrą kolejność”, skoro nie znamy wartości ruchów?

# 4. Gdy drzewo jest za duże: cztery w rzędzie *(do domu)*

Kółko i krzyżyk da się przeliczyć do końca. Ale już **cztery w rzędzie** (plansza 6 × 7) ma około $10^{21}$ możliwych partii. Pełny minimax nie ma szans.

Pomysł Shannona z wykładu:
1. **test odcięcia**: zatrzymaj się na głębokości `glebokosc`,
2. **funkcja oceny**: zamiast wypłaty zwróć szacunek, jak dobra jest pozycja dla MAX.

Funkcja oceny w grach to to samo co heurystyka $h(n)$ w A*.

### Zasady gry
- Gracze na zmianę wrzucają żeton do jednej z 7 kolumn. Żeton spada na najniższe wolne pole.
- Wygrywa ten, kto ułoży **4 żetony w linii**: poziomo, pionowo albo po skosie.
- X (MAX) zaczyna.

Stan to krotka 42 pól, wiersz 0 jest na górze. Akcja to numer kolumny 0–6.

In [ ]:
WIERSZE, KOLUMNY = 6, 7


def _linie_c4():
    linie = []
    for r in range(WIERSZE):
        for c in range(KOLUMNY):
            for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                pola = [(r + k * dr, c + k * dc) for k in range(4)]
                if all(0 <= rr < WIERSZE and 0 <= cc < KOLUMNY for rr, cc in pola):
                    linie.append(tuple(rr * KOLUMNY + cc for rr, cc in pola))
    return linie


LINIE_C4 = _linie_c4()   # wszystkie możliwe czwórki: 69 linii


class CzteryWRzedzie:
    start = ('.',) * (WIERSZE * KOLUMNY)
    KOLEJNOSC = [3, 2, 4, 1, 5, 0, 6]       # najpierw środkowe kolumny: lepsze cięcia alfa-beta

    def gracz(self, s):
        return "MAX" if s.count('X') == s.count('O') else "MIN"

    def akcje(self, s):
        return [c for c in self.KOLEJNOSC if s[c] == '.']   # kolumna niepełna, gdy górne pole wolne

    def wynik(self, s, c):
        znak = 'X' if self.gracz(s) == "MAX" else 'O'
        for r in range(WIERSZE - 1, -1, -1):
            i = r * KOLUMNY + c
            if s[i] == '.':
                return s[:i] + (znak,) + s[i + 1:]
        raise ValueError("Kolumna pełna")

    def zwyciezca(self, s):
        for a, b, c, d in LINIE_C4:
            if s[a] != '.' and s[a] == s[b] == s[c] == s[d]:
                return s[a]
        return None

    def koniec(self, s):
        return self.zwyciezca(s) is not None or '.' not in s

    def wyplata(self, s):
        # Wygrana warta 1000, plus premia za szybką wygraną (liczba wolnych pól)
        z = self.zwyciezca(s)
        wolne = s.count('.')
        if z == 'X':
            return 1000 + wolne
        if z == 'O':
            return -1000 - wolne
        return 0

    def tekst(self, s):
        wiersze = [' '.join(s[r * KOLUMNY:(r + 1) * KOLUMNY]) for r in range(WIERSZE)]
        return '\n'.join(wiersze) + '\n' + ' '.join(str(c) for c in range(KOLUMNY))


c4 = CzteryWRzedzie()
print("Liczba linii:", len(LINIE_C4))
print(c4.tekst(c4.start))

In [ ]:
def rysuj_c4(s, tytul=""):
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    ax.add_patch(plt.Rectangle((-0.5, -0.5), KOLUMNY, WIERSZE, color="#0070C0", zorder=0))
    for k, pole in enumerate(s):
        r, c = divmod(k, KOLUMNY)
        kolor = {'.': 'white', 'X': '#FFD000', 'O': '#FF0000'}[pole]
        ax.add_patch(plt.Circle((c, WIERSZE - 1 - r), 0.4, color=kolor, zorder=1))
    ax.set_xlim(-0.6, KOLUMNY - 0.4); ax.set_ylim(-0.6, WIERSZE - 0.4)
    ax.set_aspect('equal'); ax.set_xticks(range(KOLUMNY)); ax.set_yticks([])
    ax.set_title(tytul + "   (X: żółty, O: czerwony)")
    plt.show()


s = c4.start
for c in [3, 3, 4, 2]:
    s = c4.wynik(s, c)
rysuj_c4(s, "Po 4 ruchach")

### Alfa-beta z ograniczoną głębokością

To ta sama alfa-beta, z dwiema zmianami:
- parametr `glebokosc` maleje o 1 przy każdym ruchu; gdy dojdzie do 0, zwracamy `ocena(s)`,
- `ocena(s)` to funkcja oceny, którą przekazujemy jako parametr (jak `heuristic` w A* z laboratorium 1).

In [ ]:
def alfabeta_d(gra, s, glebokosc, ocena, alfa=-math.inf, beta=math.inf):
    global licznik
    licznik += 1
    if gra.koniec(s):
        return gra.wyplata(s)
    if glebokosc == 0:
        return ocena(s)                 # test odcięcia + funkcja oceny
    if gra.gracz(s) == "MAX":
        v = -math.inf
        for a in gra.akcje(s):
            v = max(v, alfabeta_d(gra, gra.wynik(s, a), glebokosc - 1, ocena, alfa, beta))
            if v >= beta:
                return v
            alfa = max(alfa, v)
        return v
    else:
        v = math.inf
        for a in gra.akcje(s):
            v = min(v, alfabeta_d(gra, gra.wynik(s, a), glebokosc - 1, ocena, alfa, beta))
            if v <= alfa:
                return v
            beta = min(beta, v)
        return v


def gracz_ab(glebokosc, ocena):
    """Tworzy gracza: alfa-beta do zadanej głębokości z daną funkcją oceny."""
    def gracz(gra, s):
        ocen = lambda g, st: alfabeta_d(g, st, glebokosc - 1, ocena)
        return najlepszy_ruch(gra, s, ocen)
    return gracz


def ocena_zero(s):
    return 0


def mecz(gra, gracz_a, gracz_b, n=6):
    """n partii; gracze zmieniają się rolami (raz A jest X, raz O). Zwraca wygrane A, remisy, wygrane B."""
    a = r = b = 0
    for i in range(n):
        if i % 2 == 0:
            w = rozegraj(gra, gracz_a, gracz_b, pokaz=False)
        else:
            w = -rozegraj(gra, gracz_b, gracz_a, pokaz=False)
        a += w > 0; r += w == 0; b += w < 0
    return a, r, b


t0 = time.time()
print("Alfa-beta (głębokość 4, ocena 0) kontra losowy:", mecz(c4, gracz_ab(4, ocena_zero), gracz_losowy))
print(f"Czas: {time.time() - t0:.1f} s")

Z oceną równą 0 algorytm widzi tylko wygrane i przegrane w zasięgu `glebokosc` ruchów. Z graczem losowym to zwykle wystarcza. Ale z mądrzejszym przeciwnikiem już nie: wszystkie „spokojne” pozycje wyglądają dla niego tak samo.

## Zadanie 2 — własna funkcja oceny *(do domu)*

Napisz funkcję `moja_ocena(s)`, która ocenia pozycję z punktu widzenia X (MAX). Wynik musi być dużo mniejszy niż 1000 (wartość wygranej).

Pomysł z wykładu (otwarte linie):
- dla każdej czwórki z `LINIE_C4` policz, ile jest w niej X i ile O,
- jeśli w czwórce są **tylko X** (i wolne pola), dodaj punkty: np. 1 za jeden X, 5 za dwa, 50 za trzy,
- jeśli są **tylko O**, odejmij punkty tak samo,
- czwórki z X i O nic nie dają (nikt w nich nie wygra).

Możesz dodać własne pomysły, np. premię za żetony w środkowej kolumnie.

In [ ]:
def moja_ocena(s):
    punkty = {0: 0, 1: 1, 2: 5, 3: 50}
    wynik = 0

    for linia in LINIE_C4:
        pola = [s[i] for i in linia]
        x = pola.count('X')
        o = pola.count('O')

        if x > 0 and o == 0:
            wynik += punkty[x]
        elif o > 0 and x == 0:
            wynik -= punkty[o]

    # Środkowa kolumna uczestniczy w największej liczbie zwycięskich linii.
    srodek = [s[r * KOLUMNY + 3] for r in range(WIERSZE)]
    wynik += 3 * (srodek.count('X') - srodek.count('O'))
    return wynik


print("Ocena pustej planszy:", moja_ocena(c4.start))
s = c4.start
for c in [3, 0, 3, 0, 3]:
    s = c4.wynik(s, c)
print("Ocena pozycji z trzema X w kolumnie:", moja_ocena(s))
assert moja_ocena(c4.start) == 0
assert 0 < moja_ocena(s) < 1000


### Sprawdź swoją funkcję oceny

Porównujemy dwa programy o **tej samej głębokości**. Różnią się tylko funkcją oceny.

In [ ]:
t0 = time.time()
wynik = mecz(c4, gracz_ab(4, moja_ocena), gracz_ab(4, ocena_zero), n=6)
print("Moja ocena kontra ocena 0 (wygrane, remisy, przegrane):", wynik)
print(f"Czas: {time.time() - t0:.1f} s")

# Zadanie 3: głębokość a jakość gry.
wynik_glebokosci = mecz(c4, gracz_ab(1, moja_ocena), gracz_ab(4, moja_ocena), n=6)
print("Głębokość 1 kontra głębokość 4 (wygrane, remisy, przegrane):", wynik_glebokosci)

wezly_dla_glebokosci = {}
for d in [2, 4, 6, 8]:
    licznik = 0
    t0 = time.time()
    gracz_ab(d, moja_ocena)(c4, c4.start)
    wezly_dla_glebokosci[d] = licznik
    print(f"głębokość {d}: " + f"{licznik:,}".replace(",", " ") + f" węzłów, {time.time() - t0:.1f} s")

print("Wniosek: większa głębokość zwykle poprawia jakość decyzji, ale liczba węzłów rośnie bardzo szybko.")
print("Efekt horyzontu występuje, gdy groźba lub korzyść leży tuż poza badaną głębokością.")


In [ ]:
# Miejsce na zadanie 3.
for d in [2, 4, 6, 8]:
    licznik = 0
    t0 = time.time()
    gracz_ab(d, moja_ocena)(c4, c4.start)
    print(f"głębokość {d}: " + f"{licznik:,}".replace(",", " ") + f" węzłów, {time.time() - t0:.1f} s")

### Zagraj z komputerem w cztery w rzędzie

Ustaw `GRAJ_C4 = True`. Wpisuj numer kolumny 0–6.

In [ ]:
GRAJ_C4 = False


def gracz_czlowiek_c4(gra, s):
    rysuj_c4(s)
    while True:
        try:
            c = int(input(f"Kolumna {sorted(gra.akcje(s))}: "))
            if c in gra.akcje(s):
                return c
        except ValueError:
            pass
        print("Niepoprawny ruch.")


if GRAJ_C4:
    w = rozegraj(c4, gracz_czlowiek_c4, gracz_ab(4, moja_ocena), pokaz=False)
    print("Wygrałeś!" if w > 0 else "Remis." if w == 0 else "Wygrał komputer.")

# 5. Zadania dla studentów

**Na zajęciach:** zadanie 1 (w treści notatnika) oraz pytania z sekcji 1–3.
**Do domu:** sekcja 4, czyli zadania 2 i 3.
**Dodatkowe:** zadania 4, 5, 6 i 7.

Po każdej zmianie w kodzie uruchom ponownie komórki od początku notatnika.

---

## Zadanie 4 — zapałki *(dodatkowe)*

Gra z wykładu: na stole leży `n` zapałek, gracze na zmianę biorą 1 albo 2, wygrywa ten, kto weźmie **ostatnią**.

Uzupełnij klasę `Zapalki`. Stan to para `(liczba_zapalek, kto_rusza)`, bo z samej liczby zapałek nie wiadomo, czyj jest ruch.
Potem użyj **tych samych** funkcji `minimax` i `wartosci_ruchow` co dla kółka i krzyżyka.

1. Czy przy 5 zapałkach pierwszy gracz wygrywa? Ile zapałek powinien wziąć?
2. Dla jakich `n` od 1 do 12 pierwszy gracz przegrywa? Jaką widzisz regułę?

In [ ]:
class Zapalki:
    def __init__(self, n=5):
        self.start = (n, "MAX")

    def gracz(self, s):
        return s[1]

    def akcje(self, s):
        return [a for a in (1, 2) if a <= s[0]]

    def wynik(self, s, a):
        nastepny = "MIN" if s[1] == "MAX" else "MAX"
        return (s[0] - a, nastepny)

    def koniec(self, s):
        return s[0] == 0

    def wyplata(self, s):
        # Gdy nie ma zapałek, wygrał gracz, który wykonał poprzedni ruch.
        return -1 if s[1] == "MAX" else 1


zap = Zapalki(5)
print("Dla 5 zapałek - wartość:", minimax(zap, zap.start))
print("Wartości ruchów:", wartosci_ruchow(zap, zap.start))

przegrywajace = []
for n in range(1, 13):
    gra_n = Zapalki(n)
    wartosc = minimax(gra_n, gra_n.start)
    if wartosc < 0:
        przegrywajace.append(n)
print("Pozycje przegrywające od 1 do 12:", przegrywajace)
print("Reguła: pierwszy gracz przegrywa dla wielokrotności 3.")


---

## Zadanie 5 — tablica transpozycji *(dodatkowe)*

Drzewo kółka i krzyżyka ma 549 946 węzłów, ale **różnych pozycji** jest dużo mniej: do tej samej pozycji można dojść różnymi kolejnościami ruchów.

Napisz `minimax_pamiec(gra, s, pamiec)`, który przed liczeniem sprawdza, czy pozycja jest już w słowniku `pamiec`, a po policzeniu zapisuje tam wynik.

1. Ile pozycji jest w słowniku po policzeniu pustej planszy? (Oczekiwane: **5 478**.)
2. Ile razy szybciej działa ta wersja?
3. Jaki element laboratorium 1 przypomina ten słownik?

In [ ]:
def minimax_pamiec(gra, s, pamiec):
    if s in pamiec:
        return pamiec[s]

    if gra.koniec(s):
        wartosc = gra.wyplata(s)
    else:
        wartosci = [minimax_pamiec(gra, gra.wynik(s, a), pamiec) for a in gra.akcje(s)]
        wartosc = max(wartosci) if gra.gracz(s) == "MAX" else min(wartosci)

    pamiec[s] = wartosc
    return wartosc


t0 = time.time()
pamiec = {}
wartosc_pamiec = minimax_pamiec(kk, kk.start, pamiec)
czas_pamiec = time.time() - t0

print("Wartość:", wartosc_pamiec)
print("Różnych pozycji:", len(pamiec))
print(f"Czas z pamięcią: {czas_pamiec:.4f} s")
print("Słownik pełni rolę tablicy transpozycji i przypomina zbiory kosztów/odwiedzonych stanów z laboratorium 1.")
assert wartosc_pamiec == 0 and len(pamiec) == 5478


---

## Zadanie 6 — Monte Carlo Tree Search *(dodatkowe)*

Poniżej gotowa implementacja MCTS (wersja UCT z wykładu: selekcja, ekspansja, symulacja, propagacja wstecz).
Nie potrzebuje funkcji oceny: wystarczą zasady gry.

1. Uruchom MCTS na pozycji-pułapce dla 100, 1 000 i 10 000 iteracji. Od ilu iteracji MCTS przestaje wybierać rogi?
2. Zmień stałą `C` (np. 0.1 i 5). Co się dzieje z rozkładem odwiedzin?
3. Napisz gracza `gracz_mcts(iteracje)` i zagraj nim w cztery w rzędzie przeciw `gracz_ab(4, moja_ocena)`.

In [ ]:
class Wezel:
    def __init__(self, gra, s, rodzic=None, ruch=None):
        self.s, self.rodzic, self.ruch = s, rodzic, ruch
        self.dzieci = []
        self.niewyprobowane = [] if gra.koniec(s) else list(gra.akcje(s))
        self.W, self.N = 0.0, 0
        self.kto_ruszyl = None if rodzic is None else gra.gracz(rodzic.s)


def mcts(gra, s, iteracje=1000, C=math.sqrt(2)):
    korzen = Wezel(gra, s)
    for _ in range(iteracje):
        w = korzen
        # 1. selekcja (UCB1)
        while not w.niewyprobowane and w.dzieci:
            w = max(w.dzieci, key=lambda d: d.W / d.N + C * math.sqrt(math.log(w.N) / d.N))
        # 2. ekspansja
        if w.niewyprobowane:
            a = w.niewyprobowane.pop(random.randrange(len(w.niewyprobowane)))
            d = Wezel(gra, gra.wynik(w.s, a), w, a)
            w.dzieci.append(d)
            w = d
        # 3. symulacja: losowa partia do końca
        st = w.s
        while not gra.koniec(st):
            st = gra.wynik(st, random.choice(gra.akcje(st)))
        u = gra.wyplata(st)
        # 4. propagacja wstecz
        while w is not None:
            w.N += 1
            if w.kto_ruszyl is not None:
                if u == 0:
                    w.W += 0.5
                elif (u > 0) == (w.kto_ruszyl == "MAX"):
                    w.W += 1
            w = w.rodzic
    return {d.ruch: d.N for d in korzen.dzieci}


for it in [100, 1000, 10000]:
    odw = mcts(kk, pulapka, it)
    rysuj_kk(pulapka, liczby=odw, tytul=f"MCTS: odwiedziny po {it} iteracjach")


def gracz_mcts(iteracje, C=math.sqrt(2)):
    def gracz(gra, s):
        odwiedziny = mcts(gra, s, iteracje=iteracje, C=C)
        najlepsza = max(odwiedziny.values())
        ruchy = [ruch for ruch, n in odwiedziny.items() if n == najlepsza]
        return random.choice(ruchy)
    return gracz


print("Wpływ stałej C dla 1000 iteracji:")
for C_test in [0.1, math.sqrt(2), 5.0]:
    random.seed(0)
    print("C =", round(C_test, 3), mcts(kk, pulapka, 1000, C=C_test))

random.seed(0)
wynik_mcts_c4 = mecz(c4, gracz_mcts(500), gracz_ab(4, moja_ocena), n=2)
print("MCTS kontra alfa-beta w cztery w rzędzie:", wynik_mcts_c4)


In [ ]:
# Zadanie 7 - przykładowy turniej funkcji oceny.
def ocena_srodek(s):
    srodek = [s[r * KOLUMNY + 3] for r in range(WIERSZE)]
    return 3 * (srodek.count('X') - srodek.count('O'))


gracze_turniej = {
    "moja_ocena": gracz_ab(4, moja_ocena),
    "ocena_zero": gracz_ab(4, ocena_zero),
    "tylko_srodek": gracz_ab(4, ocena_srodek),
}

wyniki_turnieju = []
nazwy = list(gracze_turniej)
for i in range(len(nazwy)):
    for j in range(i + 1, len(nazwy)):
        a, b = nazwy[i], nazwy[j]
        wynik = mecz(c4, gracze_turniej[a], gracze_turniej[b], n=2)
        wyniki_turnieju.append((a, b, wynik))

for a, b, wynik in wyniki_turnieju:
    print(a, "kontra", b, "->", wynik)


# 6. Podsumowanie

### Gra jako przeszukiwanie
Gra to drzewo, w którym co drugi ruch wybiera przeciwnik. Opisujemy ją funkcjami `gracz`, `akcje`, `wynik`, `koniec`, `wyplata`.

### Minimax
MAX wybiera maksimum, MIN minimum, wartości liczymy od dołu drzewa. Wynik jest optymalny, ale trzeba przejrzeć całe drzewo.

### Alfa-beta
Ten sam wynik co minimax, ale bez gałęzi, które nie mogą zmienić decyzji. Działa najlepiej przy dobrej kolejności ruchów.

### Funkcja oceny
Gdy drzewo jest za duże, zatrzymujemy się na głębokości `d` i oceniamy pozycję heurystycznie, jak $h(n)$ w A*.
Jakość gry zależy od głębokości i od jakości funkcji oceny.

# 7. Pytania kontrolne

1. Jakie funkcje opisują grę? Czym różnią się od opisu problemu z laboratorium 1?
2. Kim są MAX i MIN?
3. Jak minimax liczy wartość węzła MAX, a jak węzła MIN?
4. Dlaczego minimax nie nadaje się do szachów bez zmian?
5. Co oznaczają α i β? Kiedy następuje cięcie?
6. Czy alfa-beta może dać inny wynik niż minimax?
7. Dlaczego kolejność ruchów jest ważna dla alfa-beta?
8. Co to jest funkcja oceny i jaki ma związek z heurystyką w A*?
9. Co to jest efekt horyzontu?
10. Czym MCTS różni się od alfa-beta z funkcją oceny?